<a href="https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RodionOm/Search-ranking-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: binary classification, used as a ranking.**

Under the hood this is **binary classification** — for each page the model predicts the probability that it belongs to the "declining" class (0 to 1). But the *output is used as a ranking*: I sort pages by that probability, highest risk first, to build a review queue a human works down top-first. So the model is a classifier, but the deliverable is an ordered list — which is why the metric is a top-of-list ranking metric (Precision@K), not plain accuracy.

It is not clustering (I have a defined target, not unlabeled groups) and not pure scoring (the score is a calibrated class probability, not an arbitrary index).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess, pandas as pd

# --- ensure we can find the data (works in Colab and locally) ---
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Target or proxy

**Target (proxy): `is_declining_label = (trend_direction == "down")`.**

This is a **defined-rule proxy**, not a directly observed future outcome. It is computed from the current window's `trend_direction` field, so it labels a page as positive when its recent trend is already "down".

**Why it's a proxy, not the ideal target:** the decision I actually care about is "will this page decline *going forward*" — a future outcome. The starter label is a stand-in computed from the present window, which makes it a reasonable teaching target but a weak final one.

**Planned refinement (Week 3+, warehouse data):** move to a future-looking label — features from a prior 90-day window predicting decline over the next 30 days — with a strict leakage audit so no future information leaks into features.

**Leakage note:** because the label is derived from `trend_direction`, that column (and `trend_pct`) can never be a feature — using them would leak the answer straight into the input.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

**Metric: Precision@K (specifically Precision@50).**

**What it means:** of the top K pages the model ranks highest for review, what fraction actually carry the positive label. Precision@50 = correct picks in the top 50 ÷ 50.

**Why this metric and not accuracy:** the review team has limited capacity — they act on the top of the queue, not all 30,000 pages. So what matters is whether the *top* of the ranked list is right, not overall correctness. Accuracy would reward a model that's right on the easy majority while missing the few high-value pages at the top.

**What "good" looks like:** beat the transparent baseline rule. On the starter slice the hand-written rule scores Precision@50 = 0.240 (~12 of 50 correct); the random forest scores 0.740 (~37 of 50). "Good" = clearly and reproducibly above the 0.240 baseline, ideally near or above that ~3x lift, under honest client-holdout validation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

**What this shows:**

- **Unit of analysis:** one row = one content page (`content_id` is unique per row).
- **Target sketch:** `is_declining_label` = 1 when `trend_direction == "down"`, else 0 — the defined-rule proxy from Section 2.
- **Class balance:** ~54% positive, so the task is well-posed (not a rare-event problem). The value is in ranking the top of the queue correctly, not in raw accuracy on a near-balanced split.

Note: `trend_direction` and `trend_pct` appear here only to *derive* the label — they are excluded from any feature set, since using them would leak the answer into the model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess, pandas as pd

# --- ensure we can find the data (works in Colab and locally) ---
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git","clone","--depth","1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Unit of analysis: one row = one content page ---
print("Unit of analysis: one row = one content page")
print(f"Shape: {df.shape[0]:,} pages x {df.shape[1]} columns")
print(f"Unique content_id values: {df['content_id'].nunique():,}  "
      f"(matches row count -> one row per page)\n")

# --- Sketch the target column from the defined rule ---
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- Show the unit of analysis with a few decision-relevant columns + the target ---
cols_to_show = ["content_id", "impressions_90d", "avg_position",
                "ctr", "days_since_last_update", "trend_direction",
                "is_declining_label"]
print("One row = one page. A few signal columns plus the sketched target:")
display(df[cols_to_show].head(5))

# --- Class balance of the target ---
counts = df["is_declining_label"].value_counts().sort_index()
print(f"\nTarget balance (is_declining_label):")
print(f"  0 (not declining): {counts.get(0, 0):,}")
print(f"  1 (declining):     {counts.get(1, 0):,}")
print(f"  positive rate:     {df['is_declining_label'].mean():.1%}")

Unit of analysis: one row = one content page
Shape: 30,000 pages x 44 columns
Unique content_id values: 30,000  (matches row count -> one row per page)

One row = one page. A few signal columns plus the sketched target:


,content_id,impressions_90d,avg_position,ctr,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,3803,10.6,0.76,20,down,1
1,content_a1fb4e703a9e,15320,20.3,0.05,25,down,1
2,content_9aa793d4d895,12581,36.5,0.09,20,down,1
3,content_331d6c4de07b,11751,6.2,0.49,22,stable,0
4,content_d99b7a2d90ca,19140,44.0,0.13,14,down,1



Target balance (is_declining_label):
  0 (not declining): 13,738
  1 (declining):     16,262
  positive rate:     54.2%


## 5. Why ML beats a fixed rule here

**Why a fixed if-statement isn't enough here:**

1. **No single column carries the signal.** The obvious heuristics fail: `search_volume` barely correlates with real traffic (~0.001), and `word_count` is nearly identical for declining vs growing pages (2909 vs 2848). A rule keyed on any one column would be weak.

2. **The signal lives in combinations.** Priority comes from how many signals interact — impressions × position × freshness × CTR together — not from any one threshold. Hand-writing every interaction as nested if-statements is brittle and unmaintainable.

3. **The evidence is already in.** The starter run shows a learned model tripling the hand-rule's Precision@50 (0.240 → 0.740) on the same data — direct proof that the pattern is too multi-dimensional for a fixed rule to capture.

This is decision-support: the model *ranks candidates* for a human reviewer. It does not claim a refresh will fix a page — only which pages are worth a human's limited attention first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.